# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. All dataset elements are referenced by their Croissant schema `@id`, ensuring transparent and reproducible workflows.

### Dataset Source
The dataset is defined by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

The dataset may include multiple record sets. We use the `@id` field to reference each entity. We'll enumerate all record set and field `@id`s for inspection.

In [ ]:
# List all record sets and their fields' @id
print('Available record sets and fields:')
for record_set in metadata.record_sets:
    print(f"Record Set name: {record_set.name}, @id: {record_set.id}")
    print("  Fields (by @id and name):")
    for field in record_set.fields:
        print(f"    - {field.id} ({field.name})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references to entities (like record sets and fields) use their Croissant `@id`.

We demonstrate the process for all record sets defined in this dataset.

In [ ]:
# Extract data from available record sets
# Use the @id to select record sets programmatically

all_record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for rec_id in all_record_set_ids:
    print(f"Loading records from record set {rec_id} ...")
    records = list(dataset.records(record_set=rec_id))
    if records:
        dataframes[rec_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records into DataFrame with columns: {dataframes[rec_id].columns.tolist()}")
    else:
        print("No records loaded for this record set.")

# For demonstration, pick the first non-empty record set
selected_record_set_id = None
for rec_id, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rec_id
        break

if selected_record_set_id:
    print(f"\nPreview of data from record set {selected_record_set_id}:")
    display(dataframes[selected_record_set_id].head())
else:
    print("No non-empty record set found for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, or grouping records. All fields are referenced with their `@id`.

In [ ]:
import numpy as np

# For demonstration, select record set and a numeric field based on overview above.
# Please replace with appropriate '@id' values for your specific use-case or data field of interest.
if selected_record_set_id:
    df = dataframes[selected_record_set_id]

    # Find first numeric column for demo
    numeric_field_id = None
    for field in dataset.metadata.record_set(selected_record_set_id).fields:
        field_name = field.id
        try:
            # Simple test: see if col exists and is numeric
            if field_name in df.columns and pd.api.types.is_numeric_dtype(df[field_name]):
                numeric_field_id = field_name
                break
        except Exception:
            continue

    if numeric_field_id is not None:
        print(f"Selected numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (using column mean):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()

        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

        # Select a groupby field (first categorical field)
        group_field_id = None
        for field in dataset.metadata.record_set(selected_record_set_id).fields:
            if field.id in df.columns and pd.api.types.is_string_dtype(df[field.id]):
                group_field_id = field.id
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group_field found for demo.")
    else:
        print('No suitable numeric field found for analysis.')
else:
    print('No data to analyze in the selected record set.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we use matplotlib/seaborn for common plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and (numeric_field_id is not None) and (not filtered_df.empty):
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped_df exists
    if 'grouped_df' in locals() and group_field_id is not None and not grouped_df.empty:
        grouped_df.plot(kind='bar', legend=None, figsize=(10,4))
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric field or data for visualization.")

## 6. Conclusion
In this notebook, we've loaded the FAIR² mass spectrometry dataset using its Croissant schema, previewed the available record sets, extracted records using their `@id`, performed basic EDA (filtering, normalization, grouping), and visualized the results.

All dataset navigation and operations use schema `@id` values to ensure robust, unambiguous data processing—exemplifying FAIR and transparent data science workflows. You can further adapt this notebook for more specific research or analytics by referencing any field or record set by its unique Croissant `@id`.